In [ ]:
import random
import gym
import numpy as np
import tensorflow as tf
import keras
from collections import deque

env = gym.make('CartPole-v1')

state_shape = env.observation_space.shape[0]  
num_actions = env.action_space.n              

print(f"State Input Features : {state_shape}")
print(f"Available Actions :    {num_actions}")

memory_buffer = deque(maxlen=2000)

def build_q_network(input_dim, output_dim):
    model = keras.Sequential([
        keras.layers.Dense(24, input_dim=input_dim, activation='relu'),
        keras.layers.Dense(24, activation='relu'),
        keras.layers.Dense(output_dim, activation='linear')  
    ])
    return model

q_network = build_q_network(state_shape, num_actions)
target_network = build_q_network(state_shape, num_actions)
target_network.set_weights(q_network.get_weights())

optimizer = keras.optimizers.Adam(learning_rate=0.001)
loss_fn = tf.keras.losses.MeanSquaredError()

gamma = 0.95            
epsilon = 1.0           
epsilon_min = 0.01      
epsilon_decay = 0.995   
batch_size = 32

def choose_action(state, epsilon):
    if np.random.rand() <= epsilon:
        return env.action_space.sample()  
    
    state_tensor = tf.convert_to_tensor(state.reshape(1, -1), dtype=tf.float32)
    q_values = q_network(state_tensor)
    return int(tf.argmax(q_values, axis=1).numpy()[0])

def train_step():
    if len(memory_buffer) < batch_size:
        return 
    
    mini_batch = random.sample(memory_buffer, batch_size)
    
    states = np.array([exp[0] for exp in mini_batch])
    actions = np.array([exp[1] for exp in mini_batch])
    rewards = np.array([exp[2] for exp in mini_batch])
    next_states = np.array([exp[3] for exp in mini_batch])
    dones = np.array([exp[4] for exp in mini_batch])
    
    next_q_values = target_network(tf.convert_to_tensor(next_states, dtype=tf.float32))
    max_next_q = tf.reduce_max(next_q_values, axis=1).numpy()
    
    target_q = rewards + (1 - dones) * gamma * max_next_q
    
    with tf.GradientTape() as tape:
        current_q_predictions = q_network(tf.convert_to_tensor(states, dtype=tf.float32))
        
        one_hot_actions = tf.one_hot(actions, num_actions)
        predicted_q_values = tf.reduce_sum(current_q_predictions * one_hot_actions, axis=1)
        
        loss = loss_fn(target_q, predicted_q_values)
        
    gradients = tape.gradient(loss, q_network.trainable_variables)
    optimizer.apply_gradients(zip(gradients, q_network.trainable_variables))

num_episodes = 50 

for episode in range(num_episodes):
    state = env.reset()
    if isinstance(state, tuple): state = state[0] 
    
    total_reward = 0
    done = False
    
    while not done:
        action = choose_action(state, epsilon)
        step_results = env.step(action)
        
        next_state, reward, done = step_results[0], step_results[1], step_results[2]
        total_reward += reward
        
        memory_buffer.append((state, action, reward, next_state, done))
        state = next_state
        
        train_step()
        
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay
        
    if episode % 5 == 0:
        target_network.set_weights(q_network.get_weights())
        
    print(f"Episode : {episode+1} | Score Achieved : {total_reward} | Epsilon  : {epsilon:.3f}")

env.close()